In [ ]:
"""
Character-level GPT (nanoGPT-style) - runs end-to-end on free Colab T4 in ~10-15 min.
Paste this whole file into one Colab cell, or upload+run: !python char_gpt.py

What you'll learn: tokenization (char-level), embeddings, self-attention,
transformer blocks, training loop, checkpointing, text generation.
"""

import os
import math
import time
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------------------
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/1Chart/char_gpt_ckpt.pt"  # only used if you mount Drive
LOCAL_CKPT = "char_gpt_ckpt.pt"

# ---------------------------------------------------------------------------
# 1. Dataset - tiny Shakespeare (1MB, classic for this exercise)
# ---------------------------------------------------------------------------
data_path = "input.txt"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")

# ---------------------------------------------------------------------------
# 2. "Tokenizer" - character level. Every unique char = one token.
#    This is the simplest possible tokenizer: no BPE, no merges, just a
#    lookup table. Good for learning; real LLMs use BPE (see note at bottom).
# ---------------------------------------------------------------------------
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size} unique characters")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------------------------------------------------------
# 3. Hyperparameters - small on purpose, tuned for a fast T4 run
# ---------------------------------------------------------------------------
block_size = 256      # context length
batch_size = 64
d_model = 256
n_head = 8
n_layer = 6
dropout = 0.1
max_iters = 3000
eval_interval = 300
eval_iters = 100
learning_rate = 3e-4
ckpt_every = 500

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ---------------------------------------------------------------------------
# 4. Model - decoder-only transformer with causal self-attention
# ---------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # PyTorch's fused attention - fast, flash-attention-like, works on T4
        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm residual
        x = x + self.mlp(self.ln2(x))
        return x


class CharGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model params: {n_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------------------------------------------------------
# 5. Loss estimation helper
# ---------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------------------------------------------------------------------
# 6. Build model, optimizer, resume from checkpoint if present
# ---------------------------------------------------------------------------
model = CharGPT(vocab_size, d_model, n_head, n_layer, block_size, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

start_iter = 0
if os.path.exists(LOCAL_CKPT):
    print("Resuming from checkpoint...")
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_iter = ckpt["iter"] + 1

# ---------------------------------------------------------------------------
# 7. Training loop
# ---------------------------------------------------------------------------
t0 = time.time()
for iter in range(start_iter, max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - t0
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
              f"elapsed {elapsed/60:.1f} min")

    if iter % ckpt_every == 0 and iter > 0:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "iter": iter}, LOCAL_CKPT)
        print(f"  checkpoint saved at step {iter}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# final checkpoint
torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "iter": max_iters - 1}, LOCAL_CKPT)
print(f"\nTraining done in {(time.time()-t0)/60:.1f} minutes")

# ---------------------------------------------------------------------------
# 8. Generate sample text
# ---------------------------------------------------------------------------
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500)
print("\n--- Generated text ---\n")
print(decode(generated[0].tolist()))

Using device: cuda
Dataset length: 1,115,394 characters
Vocab size: 65 unique characters
Model params: 4.83M


KeyboardInterrupt: 

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [ ]:
"""
Character-level GPT (nanoGPT-style) - runs end-to-end on free Colab T4 in ~10-15 min.
Paste this whole file into one Colab cell, or upload+run: !python char_gpt.py

What you'll learn: tokenization (char-level), embeddings, self-attention,
transformer blocks, training loop, checkpointing, text generation.
"""

import os
import math
import time
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------------------
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/char_gpt_ckpt"  # only used if you mount Drive
LOCAL_CKPT = "char_gpt_ckpt.pt"

# ---------------------------------------------------------------------------
# 1. Dataset - tiny Shakespeare (1MB, classic for this exercise)
# ---------------------------------------------------------------------------
data_path = "input.txt"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")

# ---------------------------------------------------------------------------
# 2. "Tokenizer" - character level. Every unique char = one token.
#    This is the simplest possible tokenizer: no BPE, no merges, just a
#    lookup table. Good for learning; real LLMs use BPE (see note at bottom).
# ---------------------------------------------------------------------------
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size} unique characters")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------------------------------------------------------
# 3. Hyperparameters - small on purpose, tuned for a fast T4 run
# ---------------------------------------------------------------------------
block_size = 256      # context length
batch_size = 64
d_model = 256
n_head = 8
n_layer = 6
dropout = 0.1
max_iters = 3000
eval_interval = 300
eval_iters = 100
learning_rate = 3e-4
ckpt_every = 500

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ---------------------------------------------------------------------------
# 4. Model - decoder-only transformer with causal self-attention
# ---------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # PyTorch's fused attention - fast, flash-attention-like, works on T4
        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # pre-norm residual
        x = x + self.mlp(self.ln2(x))
        return x


class CharGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model params: {n_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------------------------------------------------------
# 5. Loss estimation helper
# ---------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------------------------------------------------------------------
# 6. Build model, optimizer, resume from checkpoint if present
# ---------------------------------------------------------------------------
model = CharGPT(vocab_size, d_model, n_head, n_layer, block_size, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

start_iter = 0
if os.path.exists(LOCAL_CKPT):
    print("Resuming from checkpoint...")
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_iter = ckpt["iter"] + 1

# ---------------------------------------------------------------------------
# 7. Training loop with progress meter (tqdm)
# ---------------------------------------------------------------------------
from tqdm import tqdm   # add this for the progress bar

t0 = time.time()
pbar = tqdm(range(start_iter, max_iters), initial=start_iter, total=max_iters, desc="Training")
for iter in pbar:
    # Evaluate loss at intervals
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - t0
        # Update progress bar description with current losses
        pbar.set_postfix({
            'train_loss': f"{losses['train']:.4f}",
            'val_loss': f"{losses['val']:.4f}",
            'elapsed': f"{elapsed/60:.1f}m"
        })

    # Checkpoint
    if iter % ckpt_every == 0 and iter > 0:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "iter": iter}, LOCAL_CKPT)
        pbar.write(f"  checkpoint saved at step {iter}")

    # Training step
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# final checkpoint
torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "iter": max_iters - 1}, LOCAL_CKPT)
print(f"\nTraining done in {(time.time()-t0)/60:.1f} minutes")

# ---------------------------------------------------------------------------
# 8. Generate sample text
# ---------------------------------------------------------------------------
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500)
print("\n--- Generated text ---\n")
print(decode(generated[0].tolist()))

Using device: cuda
Dataset length: 1,115,394 characters
Vocab size: 65 unique characters
Model params: 4.83M


Training:  17%|█▋        | 501/3000 [02:27<10:58,  3.80it/s, train_loss=2.3257, val_loss=2.3495, elapsed=1.7m]

  checkpoint saved at step 500


Training:  33%|███▎      | 1001/3000 [04:54<08:55,  3.74it/s, train_loss=1.6011, val_loss=1.7763, elapsed=4.5m]

  checkpoint saved at step 1000


...

  checkpoint saved at step 2000


Training:  83%|████████▎ | 2501/3000 [12:00<02:15,  3.68it/s, train_loss=1.2255, val_loss=1.5015, elapsed=11.6m]

  checkpoint saved at step 2500


Training: 100%|██████████| 3000/3000 [14:26<00:00,  3.46it/s, train_loss=1.1589, val_loss=1.4780, elapsed=14.4m]



Training done in 14.5 minutes

--- Generated text ---


And thou have done lovel'd us.

RICHARD:
What if thou not like with this ill thou thyself?

First Murderer:
All'though man thee which the birth true here,
In there the bloody of these bloods flowers,
I would be lurged thee stavate to thy head,
We throarge thy blood, with thee drown to them;
That would they love to quiet thy rot joy
My sisting into Thury's anon.

VALERIANA:
Out of wills deserves them shall I serve
Fall my hearts and my son oath in state,
I am none are to not borne
Off that thy ow


In [ ]:
print("\n--- Starting Chat Interface ---\n")

# Load the best checkpoint to ensure the model is fully trained
if os.path.exists(LOCAL_CKPT):
    print("Loading model from checkpoint for chat...")
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval() # Set model to evaluation mode
    print("Model loaded successfully.")
else:
    print("Warning: No checkpoint found. Using randomly initialized or partially trained model.")
    print("Please run the training cell first to train the model.")

# Simple chat loop
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print("Exiting chat.")
        break
    if not user_input.strip():
        continue

    # Encode the user's input
    # Handle characters not in vocab by ignoring them or replacing with a placeholder
    encoded_input = []
    for char in user_input:
        if char in stoi:
            encoded_input.append(stoi[char])
        else:
            # Optionally, you could use a special token for unknown characters if defined
            # For now, we'll just skip them
            print(f"Warning: Character '{char}' not in vocabulary. Skipping.")

    if not encoded_input:
        print("GPT: I couldn't understand your input (no known characters). Please try again.")
        continue

    context = torch.tensor(encoded_input, dtype=torch.long, device=device).unsqueeze(0)

    # Generate a response from the model
    # We can adjust max_new_tokens, temperature, and top_k for different behaviors
    generated_tokens = model.generate(context, max_new_tokens=100, temperature=0.9, top_k=50)

    # Decode the generated tokens (excluding the input part)
    # The model.generate returns the context + generated tokens, so we slice it
    generated_text = decode(generated_tokens[0][len(encoded_input):].tolist())

    print(f"GPT: {generated_text.strip()}")



--- Starting Chat Interface ---

Loading model from checkpoint for chat...
Model loaded successfully.
You: hi
GPT: s friar?

CAMILLO:
Were sun, sir, is your honour man: the holy is,
And should she complany to your u
You: what are you
GPT: r enemy
To unlaugh, An life, he do music come my friends.

ISABELLA:
Why, you did, as I seek; it wri
You: what
GPT: do straitiff pity:
How sounds is the father thriving slip gone!

FRIAR LAURENCE:
I would ashes hear
You: name
GPT: s and good will to seek you with such of selfs
To the absence of the straight, and thy face
To heave


KeyboardInterrupt: Interrupted by user

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


TinyStories + BPE version

In [ ]:
"""
Character-level GPT (nanoGPT-style) - runs end-to-end on free Colab T4 in ~10-15 min.

IMPORTANT: Before running, set Runtime -> Change runtime type -> T4 GPU -> Save.
This script will refuse to train on CPU by default (it's way too slow) unless
you explicitly allow it below.

Paste this whole file into one Colab cell, or upload+run: !python char_gpt.py
"""

import os
import time
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# 0. Setup + GPU check
# ---------------------------------------------------------------------------
torch.manual_seed(1337)

ALLOW_CPU = False  # set True only if you really want to train on CPU (very slow)

if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: cuda ({torch.cuda.get_device_name(0)})")
else:
    device = "cpu"
    print("WARNING: No GPU detected, using CPU.")
    print("Fix: Colab menu -> Runtime -> Change runtime type -> T4 GPU -> Save, then rerun.")
    if not ALLOW_CPU:
        raise SystemExit(
            "Stopping because no GPU is available and ALLOW_CPU=False. "
            "Switch to a GPU runtime and rerun this cell."
        )

# ---------------------------------------------------------------------------
# 1. Dataset - tiny Shakespeare (1MB, classic for this exercise)
# ---------------------------------------------------------------------------
data_path = "input.txt"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")

# ---------------------------------------------------------------------------
# 2. "Tokenizer" - character level. Every unique char = one token.
#    Simplest possible tokenizer: no BPE, no merges, just a lookup table.
#    Real LLMs use BPE - this is for learning the core architecture first.
# ---------------------------------------------------------------------------
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size} unique characters")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------------------------------------------------------
# 3. Hyperparameters
# ---------------------------------------------------------------------------
block_size = 256       # context length
batch_size = 64
d_model = 256
n_head = 8
n_layer = 6
dropout = 0.1
max_iters = 8000       # increased from 3000 - will resume from your saved checkpoint
eval_interval = 300
eval_iters = 50        # reduced from 100 for faster feedback
learning_rate = 3e-4
ckpt_every = 500

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ---------------------------------------------------------------------------
# 4. Model - decoder-only transformer with causal self-attention
# ---------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class CharGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model params: {n_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------------------------------------------------------
# 5. Loss estimation helper
# ---------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------------------------------------------------------------------
# 6. Build model, optimizer, resume from checkpoint if present
# ---------------------------------------------------------------------------
LOCAL_CKPT = "char_gpt_ckpt.pt"

model = CharGPT(vocab_size, d_model, n_head, n_layer, block_size, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

start_iter = 0
if os.path.exists(LOCAL_CKPT):
    print("Resuming from checkpoint...")
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_iter = ckpt["iter"] + 1

# ---------------------------------------------------------------------------
# 7. Training loop
# ---------------------------------------------------------------------------
t0 = time.time()
for iter in range(start_iter, max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - t0
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
              f"elapsed {elapsed/60:.1f} min")

    if iter % ckpt_every == 0 and iter > 0:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "iter": iter}, LOCAL_CKPT)
        print(f"  checkpoint saved at step {iter}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "iter": max_iters - 1}, LOCAL_CKPT)
print(f"\nTraining done in {(time.time()-t0)/60:.1f} minutes")

# ---------------------------------------------------------------------------
# 8. Generate sample text
# ---------------------------------------------------------------------------
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500)
print("\n--- Generated text ---\n")
print(decode(generated[0].tolist()))

Using device: cuda (Tesla T4)
Dataset length: 1,115,394 characters
Vocab size: 65 unique characters
Model params: 4.83M
Resuming from checkpoint...
step 3000: train loss 1.1553, val loss 1.4815, elapsed 0.1 min
  checkpoint saved at step 3000
step 3300: train loss 1.1314, val loss 1.4817, elapsed 1.5 min
  checkpoint saved at step 3500
step 3600: train loss 1.1058, val loss 1.4822, elapsed 2.8 min
step 3900: train loss 1.0769, val loss 1.4873, elapsed 4.1 min
  checkpoint saved at step 4000
step 4200: train loss 1.0447, val loss 1.5011, elapsed 5.4 min
step 4500: train loss 1.0167, val loss 1.5038, elapsed 6.7 min
  checkpoint saved at step 4500
step 4800: train loss 0.9914, val loss 1.5085, elapsed 8.0 min
  checkpoint saved at step 5000
step 5100: train loss 0.9633, val loss 1.5231, elapsed 9.3 min
step 5400: train loss 0.9332, val loss 1.5423, elapsed 10.6 min
  checkpoint saved at step 5500
step 5700: train loss 0.9069, val loss 1.5542, elapsed 11.8 min
step 6000: train loss 0.8793

TinyStories dataset + BPE

In [ ]:
"""
Word-piece (BPE) GPT trained on TinyStories - the upgrade from character-level.

Two changes from char_gpt.py:
  1. Tokenizer: real BPE (GPT-2's tokenizer via tiktoken) instead of one-char-per-token.
     This means the model works with whole word-pieces, so it can't "invent" nonsense
     words the way the character-level model did.
  2. Dataset: TinyStories - simple, modern English, much larger than Shakespeare,
     so extended training improves the model instead of causing overfitting.

Run in Colab: make sure Runtime -> T4 GPU is set, then run this whole cell/file.
First run will pip install two small packages.
"""

import os
import time
import subprocess
import sys

# ---------------------------------------------------------------------------
# 0. Install dependencies (safe to re-run, skips if already installed)
# ---------------------------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "tiktoken"], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from datasets import load_dataset

# ---------------------------------------------------------------------------
# 1. Setup + GPU check
# ---------------------------------------------------------------------------
torch.manual_seed(1337)

if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: cuda ({torch.cuda.get_device_name(0)})")
else:
    raise SystemExit(
        "No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU -> Save, then rerun."
    )

# ---------------------------------------------------------------------------
# 2. Tokenizer - GPT-2's BPE tokenizer via tiktoken (reused, not trained from
#    scratch - this is the standard "don't reinvent it" move for a first project)
# ---------------------------------------------------------------------------
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab
EOT = enc.eot_token  # end-of-text token, used to separate stories
print(f"Tokenizer vocab size: {vocab_size}")

# ---------------------------------------------------------------------------
# 3. Dataset - TinyStories, streamed subset (avoids downloading the full 2GB)
#    Increase NUM_STORIES for a better model if you have time; ~60k stories
#    keeps a full run to roughly 20-30 min on a T4.
# ---------------------------------------------------------------------------
NUM_STORIES = 60_000
CACHE_PATH = "tinystories_tokens.pt"

if os.path.exists(CACHE_PATH):
    print("Loading cached tokenized data...")
    all_ids = torch.load(CACHE_PATH)
else:
    print(f"Streaming {NUM_STORIES:,} stories from TinyStories...")
    ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

    all_ids = []
    t0 = time.time()
    for i, example in enumerate(ds):
        if i >= NUM_STORIES:
            break
        ids = enc.encode_ordinary(example["text"])
        all_ids.extend(ids)
        all_ids.append(EOT)
        if i % 10_000 == 0:
            print(f"  tokenized {i:,} stories, {len(all_ids):,} tokens so far "
                  f"({time.time()-t0:.0f}s elapsed)")

    all_ids = torch.tensor(all_ids, dtype=torch.long)
    torch.save(all_ids, CACHE_PATH)

print(f"Total tokens: {len(all_ids):,}")

n = int(0.98 * len(all_ids))
train_data = all_ids[:n]
val_data = all_ids[n:]

# ---------------------------------------------------------------------------
# 4. Hyperparameters
# ---------------------------------------------------------------------------
block_size = 256
batch_size = 16          # reduced from 64 - large vocab (50k) makes logits tensor huge
grad_accum_steps = 4     # 16 * 4 = 64 effective batch size, same as before
d_model = 384
n_head = 6
n_layer = 6
dropout = 0.1
max_iters = 4000
eval_interval = 400
eval_iters = 50
learning_rate = 3e-4
ckpt_every = 500

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ---------------------------------------------------------------------------
# 5. Model - same architecture as before, just bigger vocab + d_model
# ---------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model params: {n_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------------------------------------------------------
# 6. Loss estimation
# ---------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------------------------------------------------------------------
# 7. Build model, optimizer, resume from checkpoint if present
# ---------------------------------------------------------------------------
LOCAL_CKPT = "tinystories_gpt_ckpt.pt"

model = GPT(vocab_size, d_model, n_head, n_layer, block_size, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

start_iter = 0
best_val_loss = float("inf")
if os.path.exists(LOCAL_CKPT):
    print("Resuming from checkpoint...")
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_iter = ckpt["iter"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))

# ---------------------------------------------------------------------------
# 8. Training loop - saves a SEPARATE "best" checkpoint whenever val loss
#    improves, so you always have the best-generalizing version, even if
#    you keep training past the point where it starts overfitting.
# ---------------------------------------------------------------------------
BEST_CKPT = "tinystories_gpt_best.pt"

t0 = time.time()
for iter in range(start_iter, max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - t0
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
              f"elapsed {elapsed/60:.1f} min")
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save({"model": model.state_dict(), "iter": iter, "val_loss": best_val_loss}, BEST_CKPT)
            print(f"  new best val loss, saved to {BEST_CKPT}")

    if iter % ckpt_every == 0 and iter > 0:
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "iter": iter,
            "best_val_loss": best_val_loss,
        }, LOCAL_CKPT)

    optimizer.zero_grad(set_to_none=True)
    for micro_step in range(grad_accum_steps):
        xb, yb = get_batch("train")
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits, loss = model(xb, yb)
            loss = loss / grad_accum_steps
        loss.backward()
    optimizer.step()

torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "iter": max_iters - 1,
    "best_val_loss": best_val_loss,
}, LOCAL_CKPT)
print(f"\nTraining done in {(time.time()-t0)/60:.1f} minutes. Best val loss: {best_val_loss:.4f}")

# ---------------------------------------------------------------------------
# 9. Generate sample text using the BEST checkpoint (best generalization,
#    not necessarily the most recent/most overfit one)
# ---------------------------------------------------------------------------
print("\nLoading best checkpoint for generation...")
best = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(best["model"])
model.eval()

prompt = "Once upon a time"
context = torch.tensor([enc.encode_ordinary(prompt)], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=200)
print("\n--- Generated text ---\n")
print(enc.decode(generated[0].tolist()))

Using device: cuda (Tesla T4)
Tokenizer vocab size: 50257
Streaming 60,000 stories from TinyStories...
  tokenized 0 stories, 163 tokens so far (1s elapsed)
  tokenized 10,000 stories, 2,162,278 tokens so far (7s elapsed)
  tokenized 20,000 stories, 4,466,222 tokens so far (9s elapsed)
  tokenized 30,000 stories, 6,640,280 tokens so far (11s elapsed)
  tokenized 40,000 stories, 8,968,055 tokens so far (13s elapsed)
  tokenized 50,000 stories, 11,089,600 tokens so far (15s elapsed)
Total tokens: 13,322,817
Model params: 49.33M
step 0: train loss 10.9039, val loss 10.9055, elapsed 0.1 min
  new best val loss, saved to tinystories_gpt_best.pt
step 400: train loss 3.2093, val loss 3.2298, elapsed 10.6 min
  new best val loss, saved to tinystories_gpt_best.pt
step 800: train loss 2.7086, val loss 2.7309, elapsed 21.2 min
  new best val loss, saved to tinystories_gpt_best.pt
step 1200: train loss 2.4359, val loss 2.4744, elapsed 31.8 min
  new best val loss, saved to tinystories_gpt_best.pt


In [2]:
"""
Word-piece (BPE) GPT trained on TinyStories.
Uses the REAL TinyStories dataset via the `datasets` library (streaming + retries) -
no synthetic/dummy fallback, so you always know you're training on real data.
Includes: early stopping, live progress bar with ETA, multi-prompt generation.
"""

import os
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"

import time
import subprocess
import sys

# ---------------------------------------------------------------------------
# 0. Install dependencies
# ---------------------------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "tiktoken", "tqdm"], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from tqdm.auto import tqdm
from datasets import load_dataset

# ---------------------------------------------------------------------------
# 1. Setup + GPU check
# ---------------------------------------------------------------------------
torch.manual_seed(1337)

if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: cuda ({torch.cuda.get_device_name(0)})")
else:
    raise SystemExit(
        "No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU -> Save, then rerun."
    )

# ---------------------------------------------------------------------------
# 2. Tokenizer
# ---------------------------------------------------------------------------
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab
EOT = enc.eot_token
print(f"Tokenizer vocab size: {vocab_size}")

# ---------------------------------------------------------------------------
# 3. Dataset - REAL TinyStories only. If this fails, it fails loudly (raises),
#    rather than silently swapping in fake template data. If you hit repeated
#    failures, it's a genuine network/HF issue - wait and retry, don't proceed
#    on fake data.
# ---------------------------------------------------------------------------
NUM_STORIES = 60_000
CACHE_PATH = "tinystories_tokens.pt"   # point this at your Drive path if you want persistence

if os.path.exists(CACHE_PATH):
    print("Loading cached tokenized data...")
    all_ids = torch.load(CACHE_PATH)
else:
    print(f"Streaming {NUM_STORIES:,} stories from TinyStories...")

    max_retries = 5
    all_ids = []
    for attempt in range(1, max_retries + 1):
        try:
            ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
            all_ids = []
            t0 = time.time()
            for i, example in enumerate(ds):
                if i >= NUM_STORIES:
                    break
                ids = enc.encode_ordinary(example["text"])
                all_ids.extend(ids)
                all_ids.append(EOT)
                if i % 10_000 == 0:
                    print(f"  tokenized {i:,} stories, {len(all_ids):,} tokens so far "
                          f"({time.time()-t0:.0f}s elapsed)")
            break  # success
        except Exception as e:
            print(f"  Attempt {attempt}/{max_retries} failed: {e}")
            if attempt == max_retries:
                raise RuntimeError(
                    "Could not download real TinyStories data after several retries. "
                    "This is a Hugging Face / network issue - wait a few minutes and rerun. "
                    "Not falling back to fake data."
                )
            wait = 5 * attempt
            print(f"  Retrying in {wait}s...")
            time.sleep(wait)

    all_ids = torch.tensor(all_ids, dtype=torch.long)
    torch.save(all_ids, CACHE_PATH)

print(f"\nTotal tokens: {len(all_ids):,}")

n = int(0.98 * len(all_ids))
train_data = all_ids[:n]
val_data = all_ids[n:]

# ---------------------------------------------------------------------------
# 4. Hyperparameters
# ---------------------------------------------------------------------------
block_size = 256
batch_size = 16
grad_accum_steps = 4
d_model = 384
n_head = 6
n_layer = 6
dropout = 0.1
max_iters = 7000
eval_interval = 350
eval_iters = 50
learning_rate = 3e-4
ckpt_every = 500

# Early stopping
early_stop_patience = 5   # stop if val loss doesn't improve for 5 consecutive checks
patience_counter = 0

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ---------------------------------------------------------------------------
# 5. Model
# ---------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, d_model, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model params: {n_params/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------------------------------------------------------
# 6. Loss estimation
# ---------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------------------------------------------------------------------
# 7. Build model, optimizer, resume from checkpoint if present
# ---------------------------------------------------------------------------
LOCAL_CKPT = "tinystories_gpt_ckpt.pt"
BEST_CKPT = "tinystories_gpt_best.pt"

model = GPT(vocab_size, d_model, n_head, n_layer, block_size, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

start_iter = 0
best_val_loss = float("inf")
if os.path.exists(LOCAL_CKPT):
    print("Resuming from checkpoint...")
    ckpt = torch.load(LOCAL_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_iter = ckpt["iter"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))

# ---------------------------------------------------------------------------
# 8. Training loop - tqdm progress bar + early stopping
# ---------------------------------------------------------------------------
t0 = time.time()
current_train_loss = None
early_stop_triggered = False

pbar = tqdm(range(start_iter, max_iters), initial=start_iter, total=max_iters,
            desc="Training", unit="step")
for iter in pbar:
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        current_train_loss = losses["train"]
        elapsed = time.time() - t0
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
              f"elapsed {elapsed/60:.1f} min")

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save({"model": model.state_dict(), "iter": iter, "val_loss": best_val_loss}, BEST_CKPT)
            print(f"  new best val loss, saved to {BEST_CKPT}")
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= early_stop_patience and iter > 1000:
            print(f"\nEarly stopping: val loss hasn't improved for {early_stop_patience} checks. "
                  f"Best val loss: {best_val_loss:.4f}")
            early_stop_triggered = True
            break

    if iter % ckpt_every == 0 and iter > 0:
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "iter": iter,
            "best_val_loss": best_val_loss,
        }, LOCAL_CKPT)

    optimizer.zero_grad(set_to_none=True)
    for micro_step in range(grad_accum_steps):
        xb, yb = get_batch("train")
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits, loss = model(xb, yb)
            loss = loss / grad_accum_steps
        loss.backward()
    optimizer.step()

    pbar.set_postfix({
        "train_loss": f"{current_train_loss:.3f}" if current_train_loss is not None else "...",
        "best_val": f"{best_val_loss:.3f}" if best_val_loss < float("inf") else "...",
    })

if not early_stop_triggered:
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "iter": max_iters - 1,
        "best_val_loss": best_val_loss,
    }, LOCAL_CKPT)

print(f"\nTraining done in {(time.time()-t0)/60:.1f} minutes. Best val loss: {best_val_loss:.4f}")

# ---------------------------------------------------------------------------
# 9. Generate sample text from multiple prompts, using the BEST checkpoint
# ---------------------------------------------------------------------------
print("\nLoading best checkpoint for generation...")
best = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(best["model"])
model.eval()

prompts = ["Once upon a time", "The little girl", "One sunny day", "In the forest"]

print("\n" + "=" * 60)
print("Generated Stories")
print("=" * 60)
for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    print("-" * 40)
    context = torch.tensor([enc.encode_ordinary(prompt)], dtype=torch.long, device=device)
    generated = model.generate(context, max_new_tokens=150, temperature=0.8, top_k=40)
    print(enc.decode(generated[0].tolist()))
    print("-" * 40)

Using device: cuda (Tesla T4)
Tokenizer vocab size: 50257
Streaming 60,000 stories from TinyStories...


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

  tokenized 0 stories, 163 tokens so far (1s elapsed)
  tokenized 10,000 stories, 2,162,278 tokens so far (4s elapsed)
  tokenized 20,000 stories, 4,466,222 tokens so far (7s elapsed)
  tokenized 30,000 stories, 6,640,280 tokens so far (11s elapsed)
  tokenized 40,000 stories, 8,968,055 tokens so far (14s elapsed)
  tokenized 50,000 stories, 11,089,600 tokens so far (15s elapsed)

Total tokens: 13,322,817
Model params: 49.33M


Training:   0%|          | 0/7000 [00:00<?, ?step/s]

step 0: train loss 10.9039, val loss 10.9055, elapsed 0.1 min
  new best val loss, saved to tinystories_gpt_best.pt
step 350: train loss 3.3313, val loss 3.3351, elapsed 9.0 min
  new best val loss, saved to tinystories_gpt_best.pt
step 700: train loss 2.7746, val loss 2.7978, elapsed 18.0 min
  new best val loss, saved to tinystories_gpt_best.pt
step 1050: train loss 2.5227, val loss 2.5623, elapsed 27.0 min
  new best val loss, saved to tinystories_gpt_best.pt
step 1400: train loss 2.3273, val loss 2.3926, elapsed 36.0 min
  new best val loss, saved to tinystories_gpt_best.pt
step 1750: train loss 2.2258, val loss 2.2672, elapsed 44.9 min
  new best val loss, saved to tinystories_gpt_best.pt
step 2100: train loss 2.1273, val loss 2.1946, elapsed 54.0 min
  new best val loss, saved to tinystories_gpt_best.pt
step 2450: train loss 2.0515, val loss 2.1283, elapsed 63.0 min
  new best val loss, saved to tinystories_gpt_best.pt
step 2800: train loss 1.9583, val loss 2.0793, elapsed 72.0 m

In [3]:
import os
import shutil

SAVE_DIR = "/content/drive/MyDrive/1Chart"
os.makedirs(SAVE_DIR, exist_ok=True)

new_name = "tinystories_gpt_49M_v1.pt"
save_path = os.path.join(SAVE_DIR, new_name)

torch.save({
    "model": model.state_dict(),
    "vocab_size": vocab_size,
    "d_model": d_model,
    "n_head": n_head,
    "n_layer": n_layer,
    "block_size": block_size,
    "val_loss": best_val_loss,
}, save_path)

print(f"Saved model to {save_path}")

Saved model to /content/drive/MyDrive/1Chart/tinystories_gpt_49M_v1.pt


## GPT Chat Interface

In [4]:
print("\n--- Starting GPT Chat Interface ---\n")

# Load the best checkpoint for chat to ensure the model is fully trained and performing optimally
# We already have `best` and `model` defined from the generation step, so we can reuse them.
# If this cell were run in isolation, we would need to re-load from BEST_CKPT.

# Ensure model is in evaluation mode for inference
model.eval()
print("Model ready for chat.")

# Simple chat loop
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print("Exiting chat.")
        break
    if not user_input.strip():
        continue

    # Encode the user's input using the BPE tokenizer
    encoded_input = []
    for char_or_token in user_input:
        # Tiktoken's encode_ordinary works on strings, but if we're dealing with character-like inputs
        # it's better to ensure it's a string, or handle potential direct token IDs if that was the intent.
        # For now, assuming user_input is a string of text.
        encoded_part = enc.encode_ordinary(char_or_token)
        encoded_input.extend(encoded_part)

    if not encoded_input:
        print("GPT: I couldn't process your input. Please try again with valid text.")
        continue

    context = torch.tensor([encoded_input], dtype=torch.long, device=device)

    # Generate a response from the model
    # You can adjust max_new_tokens, temperature, and top_k for different behaviors
    generated_tokens = model.generate(context, max_new_tokens=100, temperature=0.9, top_k=50)

    # Decode the generated tokens (excluding the input part)
    # The model.generate returns the context + generated tokens, so we slice it
    generated_text = enc.decode(generated_tokens[0][len(encoded_input):].tolist())

    print(f"GPT: {generated_text.strip()}")


--- Starting GPT Chat Interface ---

Model ready for chat.
You: hello
GPT: ered into the sky. It started to rain. 

The little girl held her breath to the rain. Suddenly, a big storm came and the rain was pouring down from the sky. The rain was so strong that the rain had blown away!

The little girl's mommy was relieved and hugged her tight. "Don't forget to be careful," she said. "And no more rain!"

The little girl smiled as she watched the raindrops fall. She felt proud
You: world 
GPT: icky baa big baa-aa baa-busizzled, and baa-aa baaed all the way home. 

Afterwards, Pippin was sad to see his friend again. He had a great day playing at the park. Pippin and his friend had a great day exploring the park.<|endoftext|>Once upon a time, there was a little girl named Lily. She woke up early one morning and her mommy gave her a special bath. Lily
You: Once upon a time
GPT: ze and she felt so happy. 
Landy loved playing with her toys, especially her teddy bear. She would go outside and 

KeyboardInterrupt: Interrupted by user